# 🎭 Drama Data Processor

Notebook này dùng để:
- Chuẩn hóa dữ liệu thô từ các bài viết mạng xã hội (JSON)
- Lọc bình luận spam
- Gán nhãn dư luận bằng zero-shot classification
- Tóm tắt các nhóm ý kiến chính

👉 Sử dụng mô hình `facebook/bart-large-mnli` từ HuggingFace Transformers


In [ ]:
!pip install -q transformers accelerate tqdm


In [ ]:
import json
import os
from datetime import datetime
from tqdm import tqdm
from transformers import pipeline
import re


In [ ]:
# 👇 Đặt tên file dữ liệu đầu vào ở đây
INPUT_FILE = 'Thinh_all_posts.json'
CHECK_FILE = 'check_link.json'
OUTPUT_FILE = 'Thinh_test_data.json'

# Load dữ liệu thô
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

with open(CHECK_FILE, 'r', encoding='utf-8') as f:
    check_data = json.load(f)

print(f"Tổng số bài viết: {len(data)}")


In [ ]:
def normalize_time(original_time, url):
    check_time = check_data.get(url)
    return check_time if check_time else original_time

def merge_author(poster, author):
    return author if author else poster


In [ ]:
def is_valid_comment(text):
    if not text or len(text.strip()) < 3:
        return False
    if all(c in "😂🤣❤️👍😢🙏🔥" for c in text.strip()):
        return False
    if re.search(r'(http|www\.|t\.me|bit\.ly|linktr\.ee)', text):
        return False
    if len(text.split()) < 2:
        return False
    return True


In [ ]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["ủng hộ", "phản đối", "trung lập", "châm biếm", "khác"]

def classify_comment(text):
    result = classifier(text, labels)
    return result['labels'][0], result['scores'][0]


In [ ]:
processed = []

for post in tqdm(data):
    url = post.get("url", "")
    post["author"] = merge_author(post.get("poster", ""), post.get("author", ""))
    post["time"] = normalize_time(post.get("time", ""), url)

    valid_comments = []
    opinion_summary = {}

    for cmt in post.get("comments", []):
        if not is_valid_comment(cmt["text"]):
            continue
        label, score = classify_comment(cmt["text"])
        cmt["label"] = label
        cmt["score"] = round(score, 3)

        if label not in opinion_summary:
            opinion_summary[label] = []
        opinion_summary[label].append(cmt["text"])

        valid_comments.append(cmt)

    post["comments"] = valid_comments
    post["opinion_summary"] = {k: v[:3] for k, v in opinion_summary.items()}  # lấy 3 comment tiêu biểu
    processed.append(post)

    # Auto-save sau mỗi bài
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as out:
        json.dump(processed, out, ensure_ascii=False, indent=2)


✅ **Xử lý xong!**
- Dữ liệu đã được lưu tại `Thinh_test_data.json`
- Bạn có thể tiếp tục gộp nhiều file lại nếu cần
